# 데이터 로드 및 병합 파이프라인

GitHub에서 미리보기가 가능하도록 구성된 주피터 노트북 파일입니다.

In [ ]:
import pandas as pd
import os

DATA_DIR = "data"

## 1. 데이터 로드

In [ ]:
def load_data():
    files = {
        '기본정보': '기본정보.csv',
        '면적정보': '면적정보.csv',
        '시설정보': '시설정보.csv',
        '운영정보': '운영정보.csv',
        '위치정보': '위치정보.csv',
        '장기수선': '장기수선.csv',
        '관리비': '관리비.csv'
    }
    dfs = {}
    for name, filename in files.items():
        path = os.path.join(DATA_DIR, filename)
        try:
            dfs[name] = pd.read_csv(path, encoding='utf-8')
        except UnicodeDecodeError:
            dfs[name] = pd.read_csv(path, encoding='cp949')
        print(f"[{name}] 데이터 로드 완료 (원본): {dfs[name].shape}")
    return dfs

dfs = load_data()

## 2. 불필요한 칼럼 일괄 삭제

In [ ]:
def drop_unnecessary_columns(dfs):
    cols_to_drop = {
        '관리비': [
            '차량유지비', '지능형네트워크유지비', '재해예방비', '가스사용료(공용)', '가스사용료(전용)', 
            '기타', '제세공과금', '교육훈련비', '시설유지비', '안전점검비', '위탁관리수수료', 
            '급탕비(공용)', '수도료(공용)', 'TV수신료', '정화조오물수수료', '선관위운영비'
        ],
        '기본정보': ['시공사', '주택관리업자'],
        '시설정보': [
            '건물구조', '전기-수전용량', '전기-세대전기계약방식', '승강기관리-관리방식', 
            'CCTV대수', '부대복리시설', '홈네트워크'
        ],
        '운영정보': [
            '경비관리-계약업체', '청소관리-계약업체', '음식물 처리방법', '소독관리-계약업체', 
            '일반관리-관리방식', '경비관리-관리방식', '청소관리-관리방식', '소독관리-관리방식'
        ],
        '장기수선': ['입주자기여수익', '공동기여수익']
    }
    print("\n[진행] 불필요한 칼럼 삭제 시작...")
    for table_name, columns in cols_to_drop.items():
        if table_name in dfs:
            dfs[table_name] = dfs[table_name].drop(columns=columns, errors='ignore')
            print(f"[{table_name}] 지정된 칼럼 삭제 완료 -> 남은 컬럼 수: {dfs[table_name].shape[1]}")
    return dfs

dfs = drop_unnecessary_columns(dfs)

## 3. 데이터 병합 (Merge)

In [ ]:
def merge_data(dfs):
    if '면적정보' in dfs:
        dfs['면적정보'] = dfs['면적정보'].drop_duplicates(subset=['단지코드']).drop(columns=['주거전용면적(세부)', '세대수'], errors='ignore')

    print("\n[진행] 정적 테이블(단지코드 기준) 병합 시작...")
    static_df = dfs['기본정보']
    static_tables = ['면적정보', '시설정보', '운영정보', '위치정보']
    for table_name in static_tables:
        df_to_merge = dfs[table_name]
        cols_to_use = [col for col in df_to_merge.columns if col not in static_df.columns or col == '단지코드']
        static_df = pd.merge(static_df, df_to_merge[cols_to_use], on='단지코드', how='left')
    
    print(f"정적 데이터 병합 완료: {static_df.shape}")

    print("\n[진행] 시계열 테이블(단지코드, 발생년월 기준) 병합 시작...")
    ts_df = dfs['관리비']
    repair_df = dfs['장기수선']
    join_keys = ['단지코드', '발생년월(YYYYMM)']
    cols_to_use = [col for col in repair_df.columns if col not in ts_df.columns or col in join_keys]
    ts_df = pd.merge(ts_df, repair_df[cols_to_use], on=join_keys, how='left')
    
    print(f"시계열 데이터 병합 완료: {ts_df.shape}")

    print("\n[진행] 최종 전체 병합 (시계열 데이터 + 정적 데이터) 시작...")
    final_df = pd.merge(ts_df, static_df, on='단지코드', how='left')
    print(f"최종 Master 데이터 병합 완료: {final_df.shape}")
    return final_df

final_df = merge_data(dfs)

## 4. 최종 결과 확인

In [ ]:
# 데이터프레임 미리보기
final_df.head()

In [ ]:
# 남은 최종 컬럼 리스트
print(list(final_df.columns))